# Fig 4A, 4B, S9A (v2): ADSP + UKB meta-analysis

This version revises the ADSP and UK Biobank simulation in `12`.

- **Study LD.** `12` read `dats$UKB_Xmat` / `dats$ADSP_Xmat`, which R partial-matched to the full panels (`UKB_Xmat_full` / `ADSP_Xmat_full`), so `gen_insample` used the full-panel LD. Here `gen_insample` computes LD from the sampled rows and combines the two matrices using the realized weight w = n_ADSP / N, the weight used to generate `z_meta`.
- **Reference LD.** UKB: a fixed pool of 20,000 unrelated UKB individuals (cell 2); ADSP: all rows of the ADSP sketch. The single-panel, true-weight (`gen_refpanel`) and estimated-weight mixture fits all use these.
- **Larger study and GWAS-scale effects.** Total n = 10,000 (the ADSP sketch dimension); NCP = n × per-SNP h2g = 40, 80, 120 (as before) and 250, 500 (as the Fig S5 GWAS design). Folders are named by NCP (`ncp_<NCP>/causal_<k>/prop_<ADSP share>`); h2g = NCP / 10,000.
- **SuSiE-rRSS fits added** (`*_rrss`: `R_finite` = panel size, `eb_mix`; SER fallback model stored), next to the SuSiE-RSS fits of `12`. `manual_refpanel` (SuSiE-RSS on the estimated mixture) is kept.
- ADSP sketch columns are standardized like the UKB genotypes, so a causal variant has the same standardized effect in both cohorts; variants monomorphic in a sampled cohort are left out, and a simulation whose causal variant is monomorphic in a cohort is dropped (listed by the summary).
- **Identity-LD fits** (`identity`, `identity_rrss` with `R_finite` = 10,000, as the largest B of `28`) as a no-LD baseline.
- Simulation and fine-mapping run in one job (`[adsp_ukb_meta_v2]`), so no genotype matrix is saved. Inputs kept for later checks:
  - every output: z (meta and per cohort), the sampled row indices, the true variants, all fits, and `ld_diff` (MSE, max |ΔLD| and correlation of each reference LD and the identity against the in-sample LD, over pairs with in-sample |r| > 0.05 and over all pairs, as `28`);
  - `WORK_DIR/panel_LD/UKB_ADSP_<region>.panelLD.rds`: the two panel LDs of each region, written once (every run's `ref_order` is a subset);
  - `WORK_DIR/insample_LD/UKB_ADSP_<region>_causal_<k>_prop_<share>.insampleLD.rds`: each cohort's in-sample LD and sampled rows. The sampled rows do not depend on NCP (same seed and draws), so the NCP runs share one file; a cohort that uses its whole panel (ADSP at share 100%) is not saved, as its in-sample LD is the panel LD.
  - Rebuild the mixed LD matrices from the saved matrices and `w` or `omega_weights`. The identity matrix is implicit.

Output: `/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/ADSP_UKB_meta_v2/` (`result/`, `summary/`). `[meta_v2_summary]` here is also used by `33`.

## UKB pool: 20,000 unrelated individuals

Same procedure and seed as the 1,000-individual pool in `13` cell 2. The UKB cohort is drawn from this pool and the UKB reference LD uses all of it.

In [ ]:
src_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/ukbb_unrelated_trimed/result"
pool_n=20000
out_base="/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/ukbb_unrelated_trimed_sample_20k"
out_dir="${out_base}/result"
mkdir -p ${out_base}/{code,log} ${out_dir}
cd ${out_base}/code

seed=42     # same draw as the 1k pool of 13 cell 2; keep fixed for reproducibility

# ---- 1. one shared set of pool_n individuals (FID IID), drawn from any fileset ----
ref_prefix=$(ls ${src_dir}/*.bed | head -1 | sed 's/\.bed$//')
awk -v s=${seed} 'BEGIN{srand(s)} {print rand()"\t"$1"\t"$2}' "${ref_prefix}.fam" \
    | sort -k1,1g | head -${pool_n} | cut -f2,3 > ${out_base}/keep_pool.txt
echo "keep list: $(wc -l < ${out_base}/keep_pool.txt) individuals"

# ---- 2. list all filesets (handles gaps in the index) ----
ls ${src_dir}/*.bed | sed 's/\.bed$//' > ${out_base}/prefixes.txt
N=$(wc -l < ${out_base}/prefixes.txt)
echo "found ${N} filesets"

cat << 'EOF' > base_trim
#!/bin/bash
#BSUB -J "ukbb_pool[1-NJOBS]"
#BSUB -P acc_ad-omics
#BSUB -q express
#BSUB -n 1
#BSUB -W 1:00
#BSUB -R rusage[mem=16000]
#BSUB -o OUT_BASE/log/ukbb_pool.%J_%I.out
#BSUB -e OUT_BASE/log/ukbb_pool.%J_%I.err

module load plink/1.90
export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

in_pref=$(sed -n "${LSB_JOBINDEX}p" "OUT_BASE/prefixes.txt")
base=$(basename "${in_pref}")
plink --bfile "${in_pref}" --keep "OUT_BASE/keep_pool.txt" --keep-allele-order \
      --make-bed --out "OUT_DIR/${base}"
echo "${base}: $(wc -l < OUT_DIR/${base}.fam) samples"
EOF

sed -e "s|NJOBS|${N}|g" -e "s|OUT_BASE|${out_base}|g" -e "s|OUT_DIR|${out_dir}|g" \
    base_trim > run_ukbb_pool.sh
bsub < run_ukbb_pool.sh

In [ ]:
# checks: pool size, and ADSP sketch rows (must be >= total_n for ADSP share 100%)
head -3 /sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/ukbb_unrelated_trimed_sample_20k/result/*.summary.txt 2>/dev/null
wc -l < $(ls /sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/ukbb_unrelated_trimed_sample_20k/result/*.fam | head -1)
wc -l < $(ls /sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/ADSP_sketch_trimed/result/*.psam | head -1)   # rows + header

## Simulate and fine-map

In [ ]:
[adsp_ukb_meta_v2]
# ADSP + UKB meta-analysis (v2): simulate both cohorts and fine-map in one job, so the sampled
# genotypes never have to be saved.
#   ADSP cohort: rows of the ADSP sketch as pseudo-individuals; reference = all sketch rows.
#   UKB cohort : individuals from the UKB pool; reference = the whole pool.
#   In-sample LD = LD of the sampled rows / individuals, mixed by the realized weight
#   w = n_ADSP / N: the exact generative LD of z_meta (oracle).
#   Reference LD = each cohort's full panel, single or mixed (true or estimated weights).
#   Identity LD (SuSiE-RSS and SuSiE-rRSS with R_finite = B_identity) as a no-LD baseline.
# Per-SNP h2g = ncp / total_n.
# Inputs kept for later checks: z and sampled rows in every output; the panel LD of each region
# once in panel_ld_dir; each cohort's in-sample LD once per region / causal / prop in insample_ld_dir.
# ld_diff: MSE / max |dLD| / correlation of every reference LD (and identity) against the in-sample LD.
parameter: genofile = paths
# GTEx bim files, used only for the region index (as in 12)
parameter: cwd = path("output")
parameter: ukb_pool_dir = "/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/ukbb_unrelated_trimed_sample_20k/result"
parameter: n_causal = 1
parameter: ncp = 80
parameter: total_n = 10000
parameter: ADSP_prop = 0.5
# ADSP sample-size proportion: n_ADSP = round(ADSP_prop * total_n), n_UKB = total_n - n_ADSP
parameter: B_identity = 10000
# R_finite of the identity-LD SuSiE-rRSS fit (the largest B of 28 [cross_ref_B_sweep])
parameter: panel_ld_dir = ""
# per-region panel LD cache (written once per region if missing); "" = not saved
parameter: insample_ld_dir = ""
# in-sample LD cache, one file per region / causal / prop (shared by the ncp runs); "" = not saved
parameter: ld_thresh = 0.05
parameter: job_size = 5
parameter: walltime = "100h"
parameter: mem = "40G"
parameter: numThreads = 2
parameter: container = ""
input: genofile, group_by = 1
output: f'{cwd:a}/sample_{_index}_ncausal_{n_causal}_ncp_{ncp}_prop_{ADSP_prop}_meta.rds'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f'{step_name}_{_output[0]:bn}'
R: expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container
    library(plink2R)
    library(tidyverse)
    library(MASS)
    library(crayon)
    library(pecotmr)
    library(susieR)
    library(vroom)
    library(pgenlibr)
    library(matrixStats)
    library(Rfast)

    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/simulate_linreg.R")
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/misc.R")
    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)
    select <- dplyr::select

    # susieR GitHub build (multi-panel R, R_finite, eb_mix) in its own environment, as in 28
    source_build <- function(dir) {
      env <- new.env(parent = globalenv())
      for (f in list.files(dir, pattern = "\\.R$", full.names = TRUE)) sys.source(f, envir = env)
      env
    }
    susie_hc <- source_build("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/")$susie_rss

    out_path <- ${_output:r}
    file     <- ${_input:r}
    i        <- file %>% str_remove("\\.bim$") %>% str_remove(".*/gtex_")
    n_causal <- ${n_causal}; ncp <- ${ncp}; total_n <- ${total_n}; prop <- ${ADSP_prop}

    save_dropped <- function(reason) {
      saveRDS(list(status = "dropped", reason = reason, region = i, index = ${_index},
                   ncp = ncp, causal = n_causal, prop = "${ADSP_prop}", input_path = file), out_path)
      cat(sprintf("[DROP] region %s prop %s: %s\n", i, prop, reason))
      quit(save = "no", status = 0)
    }

    ## ---- region data: UKB pool and ADSP sketch rows, allele-aligned (as 12) ----
    UKB_data <- plink_process(file.path("${ukb_pool_dir}", paste0("UKBB_", i)))

    # ADSP pgen sketch: undo the U_MIN/U_MAX scaling used to store it
    recover_U <- function(U_scaled, Umin, Umax) {
      sweep(sweep(U_scaled, 2, 2, "/"), 2, (Umax - Umin), "*") +
        matrix(Umin, nrow(U_scaled), ncol(U_scaled), byrow = TRUE)
    }
    sketch_harmonize <- function(target_data, ref_variants, sketch) {
      qc_data <- allele_qc(match_min_prop = 0, target_data = target_data, ref_variants = ref_variants)
      qced <- qc_data$target_data_qced
      idx <- match(qced$variant_id, colnames(sketch))
      if (anyNA(idx)) stop("Some qced variant_id values not found in sketch columns.")
      sketch2 <- sketch[, idx, drop = FALSE]
      flip_index <- which(qced$variants_id_original != qced$variant_id)
      if (length(flip_index) > 0) sketch2[, flip_index] <- -sketch2[, flip_index, drop = FALSE]
      colnames(sketch2) <- qced$variants_id_original
      list(sumstat = qced, sketch = sketch2)
    }

    prefix <- paste0("/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes//ADSP_sketch_trimed/result/ADSP_", i)
    all_variant_info <- read_tsv(paste0(prefix, ".pvar"), show_col_types = FALSE)
    psam <- as.data.frame(read_tsv(paste0(prefix, ".psam"), show_col_types = FALSE))
    colnames(psam)[1:2] <- c("FID", "IID")
    pgen_obj <- pgenlibr::NewPgen(paste0(prefix, ".pgen"))
    X <- pgenlibr::ReadList(pgen_obj, variant_subset = seq_len(nrow(all_variant_info)), meanimpute = FALSE)
    rownames(X) <- psam$IID
    colnames(X) <- all_variant_info$ID
    chrom_adsp <- all_variant_info$`#CHROM`[1]
    afreq <- read_tsv(paste0("/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/rss_ld_sketch/chr", chrom_adsp,
                             "/ADSP.R4.EUR.chr", chrom_adsp, ".afreq"), show_col_types = FALSE) %>%
      slice(match(colnames(X), ID))
    variant_info <- all_variant_info
    colnames(variant_info) <- c("chrom", "pos", "original_id", "A2", "A1")

    target_tb <- UKB_data$bim %>%
      mutate(variants_id_original = paste0("chr", chrom, ":", pos, ":", A2, ":", A1),
             variant_id = variants_id_original)
    harm <- sketch_harmonize(target_data = target_tb, ref_variants = variant_info,
                             sketch = recover_U(X, afreq$U_MIN, afreq$U_MAX))

    UKB_full  <- UKB_data$bed[, colnames(harm$sketch), drop = FALSE]   # scaled by plink_process
    ADSP_full <- harm$sketch
    rm(X, harm, UKB_data); invisible(gc())

    ## ---- simulate both cohorts ----
    # variants polymorphic in both panels; ADSP columns standardized like the UKB ones so that a
    # causal variant has the same standardized effect in both cohorts
    keep0     <- colVars(UKB_full) > 0 & colVars(ADSP_full) > 0
    UKB_full  <- UKB_full[, keep0, drop = FALSE]
    ADSP_full <- scale(ADSP_full[, keep0, drop = FALSE])
    p      <- ncol(UKB_full)
    B_UKB  <- nrow(UKB_full)                    # reference size of each panel (R_finite)
    B_ADSP <- nrow(ADSP_full)

    n_ADSP <- round(prop * total_n)
    n_UKB  <- total_n - n_ADSP
    if (n_UKB > B_UKB)   stop(sprintf("n_UKB = %d exceeds the UKB pool (%d)", n_UKB, B_UKB))
    if (n_ADSP > B_ADSP) stop(sprintf("n_ADSP = %d exceeds the ADSP sketch rows (%d)", n_ADSP, B_ADSP))
    w   <- n_ADSP / total_n
    h2g <- ncp / total_n                        # per-SNP h2g (sim_multi_traits, is_h2g_total = FALSE)

    # shared causal architecture: same causal variants and effects in both cohorts and, for a
    # region, at every prop (seeded by the region index, as in 12)
    set.seed(as.integer(i))
    causal_index <- sample_causal_index(Xmat = UKB_full, ncausal = n_causal)
    B_mat <- matrix(0, nrow = p, ncol = 1)
    B_mat[, 1] <- sim_beta_fix_variant(G = UKB_full, causal_index = causal_index, is_h2g_total = FALSE)

    draw <- function(G, n) {
      if (n == 0) return(list(idx = integer(0), X = G[0, , drop = FALSE], y = numeric(0)))
      idx <- sort(sample.int(nrow(G), n))
      X <- G[idx, , drop = FALSE]
      list(idx = idx, X = X, y = as.numeric(sim_multi_traits(G = X, B = B_mat, h2g = h2g, is_h2g_total = FALSE)$P[, 1]))
    }
    UKB  <- draw(UKB_full,  n_UKB)
    ADSP <- draw(ADSP_full, n_ADSP)
    idx_UKB <- UKB$idx; idx_ADSP <- ADSP$idx              # rows of each panel used by the cohort

    # analysis variants: polymorphic in every sampled cohort
    keep <- rep(TRUE, p)
    for (co in list(UKB, ADSP)) if (length(co$idx) > 1) keep <- keep & colVars(co$X) > 0
    if (!all(keep[causal_index])) save_dropped("a causal variant is monomorphic in a sampled cohort")
    ref_order <- colnames(UKB_full)[keep]
    true_vars <- match(colnames(UKB_full)[causal_index], ref_order)

    ## ---- natural-weight meta z ----
    calc_z <- function(co) {
      if (length(co$idx) == 0) return(rep(0, length(ref_order)))
      rr <- susieR::univariate_regression(co$X[, ref_order, drop = FALSE], co$y)
      rr$betahat / rr$sebetahat
    }
    z_UKB  <- calc_z(UKB)
    z_ADSP <- calc_z(ADSP)
    z_meta <- (sqrt(n_ADSP) * z_ADSP + sqrt(n_UKB) * z_UKB) / sqrt(total_n)

    ## ---- LD ----
    ld_of <- function(G) { R <- get_cormat(G[, ref_order, drop = FALSE]); dimnames(R) <- list(ref_order, ref_order); R }
    zero  <- matrix(0, length(ref_order), length(ref_order))
    R_UKB_in   <- if (n_UKB  > 0) ld_of(UKB$X)  else zero
    R_ADSP_in  <- if (n_ADSP > 0) ld_of(ADSP$X) else zero
    R_gen_in   <- w * R_ADSP_in + (1 - w) * R_UKB_in        # in-sample, exact (oracle)
    rm(UKB, ADSP); invisible(gc())
    UKB_LD     <- ld_of(UKB_full)
    ADSP_LD    <- ld_of(ADSP_full)
    R_gen_full <- w * ADSP_LD + (1 - w) * UKB_LD            # reference panels, true weight
    I_p        <- diag(length(ref_order)); dimnames(I_p) <- dimnames(UKB_LD)

    # in-sample LD of each cohort. The sampled rows do not depend on ncp (same seed and draws), so
    # the ncp runs of a region / causal / prop share one file; a cohort that uses its whole panel is
    # not saved (its in-sample LD is the panel LD). Different rows under the same name (not expected)
    # get their own file.
    insample_ld_dir <- "${insample_ld_dir}"
    insample_path <- NULL
    if (nzchar(insample_ld_dir)) {
      insample_path <- file.path(insample_ld_dir, sprintf("UKB_ADSP_%s_causal_%s_prop_%s.insampleLD.rds", i, n_causal, "${ADSP_prop}"))
      same_rows <- function(f) { old <- readRDS(f); identical(old$idx_UKB, idx_UKB) && identical(old$idx_ADSP, idx_ADSP) }
      if (!(file.exists(insample_path) && same_rows(insample_path))) {
        if (file.exists(insample_path))
          insample_path <- sub("\\.insampleLD\\.rds$", paste0("_ncp_", ncp, ".insampleLD.rds"), insample_path)
        dir.create(insample_ld_dir, recursive = TRUE, showWarnings = FALSE)
        full <- c(UKB = n_UKB == B_UKB, ADSP = n_ADSP == B_ADSP)
        tmp  <- paste0(insample_path, ".tmp", Sys.getpid())
        saveRDS(list(region = i, causal = n_causal, prop = "${ADSP_prop}", w = w, ref_order = ref_order,
                     idx_UKB = idx_UKB, idx_ADSP = idx_ADSP, full_panel = full,
                     R_UKB_in = if (n_UKB > 0 && !full[["UKB"]]) R_UKB_in,
                     R_ADSP_in = if (n_ADSP > 0 && !full[["ADSP"]]) R_ADSP_in), tmp)
        invisible(file.rename(tmp, insample_path))
      }
    }
    rm(R_UKB_in, R_ADSP_in); invisible(gc())

    # panel LD of the region over all variants polymorphic in both panels: the same for every
    # ncp / causal / prop (each run's ref_order is a subset); written once, atomically
    panel_ld_dir <- "${panel_ld_dir}"
    if (nzchar(panel_ld_dir)) {
      panel_path <- file.path(panel_ld_dir, paste0("UKB_ADSP_", i, ".panelLD.rds"))
      if (!file.exists(panel_path)) {
        dir.create(panel_ld_dir, recursive = TRUE, showWarnings = FALSE)
        v <- colnames(UKB_full)
        ld_all <- function(G) { R <- get_cormat(G); dimnames(R) <- list(v, v); R }
        tmp <- paste0(panel_path, ".tmp", Sys.getpid())
        saveRDS(list(region = i, variant_id = v, UKB_LD = ld_all(UKB_full), ADSP_LD = ld_all(ADSP_full),
                     B_UKB = B_UKB, B_ADSP = B_ADSP), tmp)
        invisible(file.rename(tmp, panel_path))
      }
    }

    ## ---- fits: SuSiE-RSS and SuSiE-rRSS (R_finite = panel size, eb_mix) ----
    fit <- function(tag, ...) tryCatch(
      suppressWarnings(susie_hc(z = z_meta, n = total_n, ...)),
      error = function(e) { cat(sprintf("[FAIL-FIT] region %s prop %s %s: %s\n", i, prop, tag, conditionMessage(e))); NULL })
    rrss <- function(tag, R, B) fit(tag, R = R, R_finite = B, R_mismatch = "eb_mix")

    result_list <- list(
      gen_insample       = fit("gen_insample", R = R_gen_in),
      gen_refpanel       = fit("gen_refpanel", R = R_gen_full),
      UKB_refpanel       = fit("UKB_refpanel", R = UKB_LD),
      ADSP_refpanel      = fit("ADSP_refpanel", R = ADSP_LD),
      mix_refpanel       = fit("mix_refpanel", R = list(ADSP_LD, UKB_LD)),      # omega[1] = ADSP, [2] = UKB
      UKB_refpanel_rrss  = rrss("UKB_refpanel_rrss", UKB_LD, B_UKB),
      ADSP_refpanel_rrss = rrss("ADSP_refpanel_rrss", ADSP_LD, B_ADSP),
      mix_refpanel_rrss  = rrss("mix_refpanel_rrss", list(ADSP_LD, UKB_LD), c(B_ADSP, B_UKB)),
      identity           = fit("identity", R = I_p),
      identity_rrss      = rrss("identity_rrss", I_p, ${B_identity})
    )
    om      <- result_list$mix_refpanel$omega_weights
    om_rrss <- result_list$mix_refpanel_rrss$omega_weights
    mixed   <- function(o) if (length(o) == 2) o[1] * ADSP_LD + o[2] * UKB_LD else NULL
    result_list$manual_refpanel <- if (!is.null(mixed(om))) fit("manual_refpanel", R = mixed(om))

    # largest |r| between the top variants of different CSs, against the LD each fit used
    # (identity fits: against the in-sample LD, since the identity has no off-diagonal LD)
    cross_cs_maxcorr <- function(fit, R) {
      cs <- fit$sets$cs
      if (is.null(fit) || is.null(R) || length(cs) < 2) return(NA_real_)
      tops <- vapply(cs, function(m) m[which.max(fit$pip[m])], integer(1))
      cc <- abs(R[tops, tops]); diag(cc) <- NA
      max(cc, na.rm = TRUE)
    }
    R_used <- list(gen_insample = R_gen_in, gen_refpanel = R_gen_full,
                   UKB_refpanel = UKB_LD, ADSP_refpanel = ADSP_LD, mix_refpanel = mixed(om),
                   UKB_refpanel_rrss = UKB_LD, ADSP_refpanel_rrss = ADSP_LD, mix_refpanel_rrss = mixed(om_rrss),
                   manual_refpanel = mixed(om), identity = R_gen_in, identity_rrss = R_gen_in)
    max_cs_corr <- imap_dbl(result_list, function(f, nm) if (is.null(f)) NA_real_ else cross_cs_maxcorr(f, R_used[[nm]]))

    ## ---- LD difference of each reference LD from the in-sample LD (metrics as 28 [cross_ref_ld_similarity]) ----
    ld_metrics <- function(R_ref, R_true, filter) {
      ut <- upper.tri(R_true); a <- R_ref[ut]; b <- R_true[ut]
      ok <- is.finite(a) & is.finite(b)
      if (filter) ok <- ok & abs(b) > ${ld_thresh}
      if (!any(ok)) return(tibble(mse = NA_real_, max_abs_diff = NA_real_, concord = NA_real_, n_pairs = 0L))
      d <- a[ok] - b[ok]
      tibble(mse = mean(d^2), max_abs_diff = max(abs(d)),
             concord = if (sum(ok) > 1 && sd(a[ok]) > 0) suppressWarnings(cor(a[ok], b[ok])) else NA_real_,
             n_pairs = sum(ok))
    }
    ld_refs <- compact(list(gen_refpanel = R_gen_full, UKB_refpanel = UKB_LD, ADSP_refpanel = ADSP_LD,
                            mix_refpanel = mixed(om), mix_refpanel_rrss = mixed(om_rrss), identity = I_p))
    ld_diff <- imap_dfr(ld_refs, function(R, ref)
      map_dfr(setNames(c(TRUE, FALSE), c("insample_r_gt_${ld_thresh}", "all")), function(flt) ld_metrics(R, R_gen_in, flt), .id = "pairs") %>%
        mutate(reference = ref, .before = 1))

    output <- list(
      status = "ok", design = "v2",
      result_list = result_list,
      region = i, index = ${_index}, input_path = file,
      ncp = ncp, h2g = h2g, causal = n_causal, prop = "${ADSP_prop}", w = w,
      n_sample = total_n, total_n = total_n, n_UKB = n_UKB, n_ADSP = n_ADSP,
      B_UKB = B_UKB, B_ADSP = B_ADSP,
      ref_order = ref_order, true_vars = true_vars,          # true_vars index ref_order, as the CSs do
      z_meta = z_meta, z_UKB = z_UKB, z_ADSP = z_ADSP,
      idx_UKB = idx_UKB, idx_ADSP = idx_ADSP,
      omega_weights = om, omega_weights_rrss = om_rrss,
      max_cs_corr = max_cs_corr, ld_diff = ld_diff,
      panel_ld_path = if (nzchar(panel_ld_dir)) panel_path, insample_ld_path = insample_path
    )
    dir.create(dirname(out_path), recursive = TRUE, showWarnings = FALSE)
    saveRDS(output, out_path)

    ncs <- function(f) if (is.null(f)) NA else length(f$sets$cs)
    cat(sprintf("[OK] region %s prop %s ncp %s: CS in-sample=%s ref=%s UKB=%s ADSP=%s mix=%s mix_rrss=%s; w=%.2f omega=%s\n",
                i, prop, ncp, ncs(result_list$gen_insample), ncs(result_list$gen_refpanel),
                ncs(result_list$UKB_refpanel), ncs(result_list$ADSP_refpanel), ncs(result_list$mix_refpanel),
                ncs(result_list$mix_refpanel_rrss), w, paste(round(om, 2), collapse = "/")))

In [ ]:
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output"
job_name="ADSP_UKB_meta_v2"
job_dir="${base_dir}/${job_name}"
mkdir -p "${job_dir}"/{code,result,log}
cd "${job_dir}/code"

cat << 'EOF' > ${job_name}.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME_CAUSAL_NCP_PROP
#BSUB -n 5
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=60000]
#BSUB -W 90:00
#BSUB -o WORK_DIR/log/JOB_NAME_CAUSAL_NCP_PROP.%J.out
#BSUB -e WORK_DIR/log/JOB_NAME_CAUSAL_NCP_PROP.%J.err
export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/susie-rRSS-paper/simulation_code/2026_Simulation_Sinai/32.Fig4A_4B_S9A.ADSP_UKB_Meta_v2.ipynb" adsp_ukb_meta_v2 \
    --genofile $(echo /sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/gtex_trimed_random/*.bim) \
    --n_causal CAUSAL --ncp NCP --ADSP_prop PROP --total_n 10000 \
    --panel_ld_dir WORK_DIR/panel_LD --insample_ld_dir WORK_DIR/insample_LD \
    --cwd WORK_DIR/result/ncp_NCP/causal_CAUSAL/prop_PROP/ \
    --mem 40G -j 5
EOF
chmod +x ${job_name}.sh

for causal in 1 2 3; do
  for ncp in 80 120 250 500; do
    for prop in 0.0 0.20 0.50 0.80 1.0; do
      output_script="${job_name}_causal_${causal}_ncp_${ncp}_prop_${prop}.sh"
      sed -e "s|WORK_DIR|${job_dir}|g" -e "s|JOB_NAME|${job_name}|g" \
          -e "s|CAUSAL|${causal}|g" \
          -e "s|NCP|${ncp}|g" \
          -e "s|PROP|${prop}|g" \
          ${job_name}.sh > "${output_script}"
      sb "${output_script}"
    done
  done
done

## Summary

Power / FDR counts per fit (including the identity fits), the mixture weights, and the SuSiE-rRSS diagnostics (flag, `Q_art`, `D_CS,max`) with `<fit>_SER` rows for the fallback model; the LD differences of every run go to `<summary>_ld_diff.rds`.

In [ ]:
[meta_v2_summary]
# One row per (simulation, fit), plus "<fit>_SER" rows for the SER fallback model of each SuSiE-rRSS
# fit; for the outputs of 32 [adsp_ukb_meta_v2] and 33 [afr_eur_meta_v2]. Dropped simulations are
# listed in <output>_dropped.rds, and each run's LD differences from the in-sample LD in <output>_ld_diff.rds.
parameter: folder = path
parameter: cwd = path("output")
parameter: job_size = 1
parameter: walltime = "50h"
parameter: mem = "30G"
parameter: numThreads = 4
parameter: container = ""
output: f'{cwd:a}/{folder:b}_{step_name}.rds'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f'{step_name}_{_output[0]:bn}'
R: expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container
    library(tidyverse)
    library(parallel)
    `%||%` <- function(a, b) if (is.null(a)) b else a

    true_set_num <- function(variant, cs) {
      if (length(cs) == 0) return(0L)
      sum(vapply(cs, function(x) length(intersect(variant, x)) > 0, logical(1)))
    }
    discover_var_num <- function(variant, cs) {
      if (length(cs) == 0) return(0L)
      length(intersect(variant, unlist(cs)))
    }

    files <- list.files(${folder:r}, pattern = "sample_.*_meta\\.rds$", recursive = TRUE, full.names = TRUE)

    read_one <- function(f) {
      res <- readRDS(f)
      if (!identical(res$status, "ok"))
        return(list(dropped = tibble(input_file = f, reason = res$reason %||% NA_character_,
                                     ncp = res$ncp %||% NA_real_, causal = res$causal %||% NA_integer_,
                                     prop = as.character(res$prop %||% NA))))
      variant <- res$true_vars
      om <- res$omega_weights; om_r <- res$omega_weights_rrss
      row_from_fit <- function(fit, label, diag) {
        cs <- if (is.null(fit)) list() else fit$sets$cs
        tibble(
          input_file = f, method = label, fit_ok = !is.null(fit),
          ncp = res$ncp, h2g = as.character(res$h2g), causal = res$causal, index = res$index,
          region = res$region, prop = as.character(res$prop), w = res$w,
          n_sample = res$n_sample, total_n = res$total_n,
          omega_1 = if (length(om) == 2) om[1] else NA_real_,
          omega_2 = if (length(om) == 2) om[2] else NA_real_,
          omega_rrss_1 = if (length(om_r) == 2) om_r[1] else NA_real_,
          omega_rrss_2 = if (length(om_r) == 2) om_r[2] else NA_real_,
          true_var_number  = discover_var_num(variant, cs),
          true_set_number  = true_set_num(variant, cs),
          total_var_number = length(variant),
          total_set_number = length(cs),
          total_set_length = length(unlist(cs)),
          max_cs_corr = unname(res$max_cs_corr[label] %||% NA_real_),
          R_reliability_flag = diag$R_reliability_flag %||% NA,
          Q_art    = diag$Q_art %||% NA_real_,
          D_CS_max = if (length(diag$bf_attenuation$cs_max))
                       suppressWarnings(max(diag$bf_attenuation$cs_max, na.rm = TRUE)) else NA_real_
        )
      }
      rows <- imap_dfr(res$result_list, function(fit, m) {
        diag <- fit$R_finite_diagnostics
        out  <- row_from_fit(fit, m, diag)
        if (!is.null(diag$ser_model))
          out <- bind_rows(out, row_from_fit(diag$ser_model, paste0(m, "_SER"), diag$ser_model$R_finite_diagnostics) %>%
                                  mutate(R_reliability_flag = diag$R_reliability_flag %||% NA))
        out
      })
      ld_diff <- if (!is.null(res$ld_diff))
        res$ld_diff %>% mutate(input_file = f, ncp = res$ncp, causal = res$causal, index = res$index,
                               region = res$region, prop = as.character(res$prop), w = res$w, .before = 1)
      list(rows = rows, ld_diff = ld_diff)
    }

    parts <- mclapply(files, function(f) tryCatch(read_one(f), error = function(e) {
      message("FAILED: ", f, " : ", conditionMessage(e)); NULL }), mc.cores = ${numThreads})
    tb      <- bind_rows(map(parts, "rows"))
    dropped <- bind_rows(map(parts, "dropped"))
    ld_diff <- bind_rows(map(parts, "ld_diff"))
    saveRDS(tb, ${_output:r})
    saveRDS(dropped, sub("\\.rds$", "_dropped.rds", ${_output:r}))
    saveRDS(ld_diff, sub("\\.rds$", "_ld_diff.rds", ${_output:r}))
    cat(sprintf("%d files: %d fit rows, %d dropped simulations\n", length(files), nrow(tb), nrow(dropped)))

In [ ]:
work_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/ADSP_UKB_meta_v2"
mkdir -p ${work_dir}/{summary,log}
cd ${work_dir}/summary

cat << 'EOF' > summary.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J ADSP_UKB_meta_v2_summary
#BSUB -n 4
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=40000]
#BSUB -W 16:00
#BSUB -o WORK_DIR/log/summary.%J.out
#BSUB -e WORK_DIR/log/summary.%J.err
export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/susie-rRSS-paper/simulation_code/2026_Simulation_Sinai/32.Fig4A_4B_S9A.ADSP_UKB_Meta_v2.ipynb" meta_v2_summary \
    --folder WORK_DIR/result/ \
    --cwd WORK_DIR/summary --mem 40G
EOF
sed "s|WORK_DIR|${work_dir}|g" summary.sh > summary_submit.sh
sb summary_submit.sh